# Advanced EDA: Change Over Time Analysis

## Executive Summary & Analytical Domain
This notebook performs comprehensive **Time-Series Analysis** on historical enterprise sales transactions from the Medallion Gold star schema (`gold.fact_sales`). Tracking metric trajectories across calendar intervals is essential for:
- Evaluating enterprise growth rates, revenue velocity, and sales momentum.
- Uncovering seasonal cycles, holiday demand spikes, and off-peak troughs.
- Measuring customer acquisition dynamics and transaction volume over time.

## Key Performance Indicators (KPIs) & Methodology
- **Total Revenue (`SUM(sales_amount)`)**: Core top-line financial performance.
- **Active Customers (`COUNT(DISTINCT customer_key)`)**: Buyer engagement and customer reach.
- **Total Units Sold (`SUM(quantity)`)**: Physical product demand velocity.
- **Temporal Granularities**:
  1. *Annual Horizons*: Long-term macro growth trajectories (`extract(year from order_date)`).
  2. *Monthly Horizons*: Continuous operational time-series (`date_trunc('month', order_date)`).
  3. *Formatted Calendars*: Human-readable monthly cadences (`date_format(order_date, 'yyyy-MMM')`).
  4. *Multi-Tier Intervals*: Year-by-month hierarchical breakdowns.
  5. *Annual Cohorts*: Focused deep-dive on historical benchmark years (e.g., 2011).

## Data Contracts & Layer Dependencies
- **Target Layer**: `gold`
- **Primary Source Table**: `gold.fact_sales` (`order_date`, `sales_amount`, `quantity`, `customer_key`)

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Setup display helper and session verification
try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure gold database context
spark.sql("USE gold")
print("Session configured. Active database context: gold")

## 1. Yearly Sales Performance Analysis

### Business Context & Objective
Annual aggregation provides high-level visibility into macro business trends, allowing leadership to evaluate year-over-year revenue stability, customer base growth, and total unit sales.

In [ ]:
# COMMAND ----------
# 1. Yearly Analysis: Sales, Customer Base, and Quantity
df_yearly = spark.sql("""
SELECT
    extract(year from order_date) AS order_year,
    sum(sales_amount) AS total_sales,
    count(distinct customer_key) AS total_customers,
    sum(quantity) AS total_quantity
FROM gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY extract(year from order_date)
ORDER BY order_year
""")
display(df_yearly)

### Strategic Insights & Interpretation (Yearly)
- **Revenue Trajectory**: Identifies whether top-line sales growth is expanding at a sustained rate.
- **Customer Base Expansion**: Correlates customer count growth with total sales to evaluate if revenue gains stem from new customer acquisition or higher spend per customer.
- **Order Volume**: Tracks physical product units shipped over successive operating years.

## 2. Monthly Sales Trends (Continuous Time-Series via `date_trunc`)

### Business Context & Objective
Monthly truncation standardizes each transaction to the first day of its respective month, producing an unbroken continuous timeline for operational monitoring and trend analysis.

In [ ]:
# COMMAND ----------
# 2. Monthly Analysis: Continuous Timeline via date_trunc
df_monthly_trunc = spark.sql("""
SELECT
    date_trunc('month', order_date) AS order_month,
    sum(sales_amount) AS total_sales,
    count(distinct customer_key) AS total_customers,
    sum(quantity) AS total_quantity
FROM gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY date_trunc('month', order_date)
ORDER BY order_month
""")
display(df_monthly_trunc)

### Strategic Insights & Interpretation (Monthly Timeline)
- **Seasonality & Cyclicality**: Reveals recurring monthly patterns, holiday season surges (Q4 peaks), and summer slowdowns.
- **Operational Volatility**: Highlights sudden shifts or anomalies in order volumes and customer participation.

## 3. Formatted Monthly Analysis (`date_format`)

### Business Context & Objective
Formats date timestamps into standardized calendar month strings (`yyyy-MMM`) for executive reporting presentations and dashboarding tools.

In [ ]:
# COMMAND ----------
# 3. Monthly Analysis: Formatted Output (yyyy-MMM)
df_monthly_formatted = spark.sql("""
SELECT
    date_format(order_date, 'yyyy-MMM') AS order_month,
    sum(sales_amount) AS total_sales,
    count(distinct customer_key) AS total_customers,
    sum(quantity) AS total_quantity
FROM gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY date_format(order_date, 'yyyy-MMM')
ORDER BY order_month
""")
display(df_monthly_formatted)

### Strategic Insights & Interpretation (Formatted Cadence)
- Formats dates using Databricks-native `date_format(order_date, 'yyyy-MMM')` for standardized month-year grouping.
- Produces clean categorical labels for BI chart visualization.

## 4. Multi-Tier Interval Analysis (Year & Month Breakdown)

### Business Context & Objective
Decomposes temporal metrics simultaneously by calendar year and numerical month (1-12) to compare seasonal behavior across parallel calendar years.

In [ ]:
# COMMAND ----------
# 4. Multi-Tier Temporal Breakdown: Year & Month
df_year_month = spark.sql("""
SELECT
    extract(year from order_date) AS order_years,
    extract(month from order_date) AS order_months,
    sum(sales_amount) AS total_sales,
    count(distinct customer_key) AS total_customers,
    sum(quantity) AS total_quantity
FROM gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY extract(year from order_date), extract(month from order_date)
ORDER BY order_years, order_months
""")
display(df_year_month)

### Strategic Insights & Interpretation (Multi-Tier Breakdown)
- Facilitates comparative monthly analysis across multiple years (e.g., comparing June 2011 to June 2012).
- Serves as the foundation for calculating Month-over-Month (MoM) and Year-over-Year (YoY) percentage changes.

## 5. Annual Cohort Deep-Dive: 2011 Sales Seasonality

### Business Context & Objective
Performs an in-depth seasonal breakdown for operating year 2011, detailing intra-year demand fluctuations across all twelve calendar months.

In [ ]:
# COMMAND ----------
# 5. Cohort Analysis: 2011 Monthly Seasonality Breakdown
df_2011_cohort = spark.sql("""
SELECT
    extract(year from order_date) AS order_years,
    extract(month from order_date) AS order_months,
    date_format(order_date, 'yyyy-MMM') AS month_name,
    sum(sales_amount) AS total_sales,
    count(distinct customer_key) AS total_customers,
    sum(quantity) AS total_quantity
FROM gold.fact_sales
WHERE order_date IS NOT NULL
  AND extract(year from order_date) = 2011
GROUP BY extract(year from order_date), extract(month from order_date), date_format(order_date, 'yyyy-MMM')
ORDER BY order_years, order_months
""")
display(df_2011_cohort)

### Business Recommendations & Summary
1. **Demand Forecasting**: Align supply chain replenishment schedules with historical seasonal sales surges.
2. **Promotional Timing**: Schedule promotional campaigns during historically low-volume months to stimulate counter-cyclical revenue.
3. **Capacity Allocation**: Ensure warehouse and logistics fulfillment capacity is scaled up ahead of high-volume Q4 periods.